# Pathway 입력 방식 비교: MLP 세 조건
구조 입력은 모두 켭니다. pathway 없음 / 평균 점수 / 개별 probe를 받는 sparse encoder를 같은 분할에서 비교합니다.

Sparse encoder는 KEGG 소속 연결만 허용하고, 연결별 가중치를 학습합니다. pathway마다 4개 노드를 사용한 뒤 32차원으로 압축합니다. 원래 발현 branch도 유지합니다. 평균 입력 대조군은 같은 encoder에 pathway 평균 점수를 넣습니다. 이전 평균 branch와 architecture가 달라 세 모델을 모두 새로 학습합니다. 명목상 parameter 수는 같지만 pathway 없음에서는 해당 branch가 비활성입니다.

평균과 달리 반대 방향의 유전자 패턴을 구분할 수 있지만, 정보의 완전한 보존을 보장하지는 않습니다. 여러 probe가 한 유전자에 매핑되면 개별 probe 연결을 사용합니다. P-NET의 sparse 연결 원리를 참고한 탐색 모델이며 해당 논문을 재현한 것은 아닙니다.

학습 데이터로만 표준화를 적합합니다. 내부 검증으로 epoch를 선택하고 외부 학습 데이터로 재학습합니다. 기존과 같은 seed 42 화합물 분할이며 반복해서 살펴본 탐색 결과입니다. 최종 독립 검증으로 해석하지 마세요.

새 커널에서 Run All. 아래 경로를 확인하세요. 기본 30 epoch × 내부/재학습 × 3모델이며 진행 상황을 출력합니다. 중단 후 재시작하면 새 폴더에서 다시 학습합니다.

In [ ]:
"""First-meeting experiment: identical MLP, with/without own Morgan features."""
import argparse
import copy
import json
from pathlib import Path
import random
import joblib
import numpy as np
import pandas as pd
import torch
from torch import nn
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from threadpoolctl import threadpool_limits


def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)


def condition_features(pairs):
    result={}
    for suffix in ['source','target']:
        result[f'{suffix}_dose']=np.log1p(pairs[f'dose_{suffix}'].to_numpy(float))
        result[f'{suffix}_time']=pairs[f'time_h_{suffix}'].to_numpy(float)
        for key in ['unit','level']:result[f'{suffix}_{key}']=pairs[f'{key}_{suffix}'].astype(str).to_numpy()
    return pd.DataFrame(result)


class Data:
    def __init__(self,a):
        self.pairs=pd.read_csv(a.prepared/'pairs.csv.gz')
        self.source=np.load(a.prepared/'vitro_expression.npy')
        self.target=np.load(a.prepared/'vivo_expression.npy')
        self.vo=pd.read_csv(a.prepared/'vivo_conditions.csv')
        probes=pd.read_csv(a.prepared/'probes.csv')['probe']
        if probes.duplicated().any() or self.source.shape[1]!=len(probes) or self.target.shape[1]!=len(probes):raise ValueError('Probe dimension mismatch')
        for name,matrix in [('vitro',self.source),('vivo',self.target)]:
            table=pd.read_csv(a.prepared/f'{name}_conditions.csv')
            if not np.array_equal(table.condition_id,np.arange(len(matrix))) or not np.isfinite(matrix).all():raise ValueError('Invalid condition-expression alignment')
            suffix='source' if name=='vitro' else 'target'
            ids=self.pairs[f'condition_id_{suffix}'].to_numpy(int)
            if (ids<0).any() or (ids>=len(matrix)).any():raise ValueError('Condition ID range error')
            if not np.array_equal(table.iloc[ids].compound.to_numpy(),self.pairs.compound.to_numpy()):raise ValueError('Compound mismatch')
        if not self.pairs.level_target.eq('Control').any():raise ValueError('Use prepared_controls including target controls')
        if self.pairs.duplicated(['condition_id_source','condition_id_target']).any():raise ValueError('Duplicate source-target pair')
        self.features=condition_features(self.pairs)
        st=pd.read_csv(a.structure);st.compound=st.compound.astype(str).str.strip()
        if st.compound.duplicated().any():raise ValueError('Duplicate structure names')
        st=st.set_index('compound')
        if not np.isfinite(st.to_numpy(float)).all() or not np.isin(st.to_numpy(),[0,1]).all():raise ValueError('Binary Morgan fingerprint required')
        st=st.copy();st['structure_present']=1.
        self.names=sorted(self.pairs.compound.unique())
        self.structure=st.reindex(self.names).fillna(0).to_numpy(np.float32)
        self.compound_to_index={n:i for i,n in enumerate(self.names)}
        self.compound_indices=self.pairs.compound.map(self.compound_to_index).to_numpy(int)
        self.structure_audit=pd.DataFrame({'compound':self.names,'structure_present':self.structure[:,-1].astype(int)})
        self.source_ids=self.pairs.condition_id_source.to_numpy(int)
        self.target_ids=self.pairs.condition_id_target.to_numpy(int)


def prepare(data,tr,ev):
    unique_source=np.unique(data.source_ids[tr]);unique_target=np.unique(data.target_ids[tr])
    source_scale=StandardScaler().fit(data.source[unique_source])
    source=source_scale.transform(data.source).astype(np.float32)
    numeric=[c for c in data.features if c.endswith(('_dose','_time'))]
    category=[c for c in data.features if c not in numeric]
    encoder=ColumnTransformer([('numeric',StandardScaler(),numeric),('category',OneHotEncoder(handle_unknown='ignore',sparse_output=False),category)])
    encoder.fit(data.features.iloc[tr])
    conditions=encoder.transform(data.features).astype(np.float32)
    # Centering only: retain original probe-space MSE and output all probes directly.
    target_mean=data.target[unique_target].mean(0).astype(np.float32)
    _,inverse,counts=np.unique(data.target_ids[tr],return_inverse=True,return_counts=True)
    weights=(1/counts[inverse]).astype(np.float32);weights=weights/weights.mean()
    return dict(source=torch.tensor(source),conditions=torch.tensor(conditions),structure=torch.tensor(data.structure),
        target=torch.tensor(data.target-target_mean),target_mean=target_mean,weights=torch.tensor(weights),
        preprocess=dict(source_scale=source_scale,condition_encoder=encoder,target_mean=target_mean))


class MLP(nn.Module):
    def __init__(self,input_dim,output_dim):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(input_dim,128),nn.SiLU(),nn.Dropout(.1),nn.Linear(128,64),nn.SiLU(),nn.Linear(64,output_dim))
    def forward(self,source,condition,structure):
        return self.net(torch.cat([source,condition,structure],dim=1))


def inputs(data,pack,rows,use_structure):
    source=pack['source'][data.source_ids[rows]]
    condition=pack['conditions'][rows]
    structure=pack['structure'][data.compound_indices[rows]]
    if not use_structure:structure=torch.zeros_like(structure)
    return source,condition,structure


def model_for(data,pack,seed):
    seed_all(seed)
    return MLP(pack['source'].shape[1]+pack['conditions'].shape[1]+pack['structure'].shape[1],data.target.shape[1])


def train_epoch(model,opt,data,pack,tr,use_structure,batch_size):
    model.train();total=0.
    for local in torch.randperm(len(tr)).split(batch_size):
        rows=tr[local.numpy()]
        pred=model(*inputs(data,pack,rows,use_structure))
        loss=((pred-pack['target'][data.target_ids[rows]]).square().mean(1)*pack['weights'][local]).mean()
        if not torch.isfinite(loss):raise ValueError('Nonfinite training loss')
        opt.zero_grad();loss.backward();nn.utils.clip_grad_norm_(model.parameters(),1.);opt.step()
        total+=loss.item()*len(rows)
    return total/len(tr)


@torch.no_grad()
def predict(model,data,pack,rows,use_structure,batch_size):
    model.eval();ids=np.unique(data.target_ids[rows]);lookup={int(cid):i for i,cid in enumerate(ids)}
    sums=np.zeros((len(ids),data.target.shape[1]),np.float64);counts=np.zeros(len(ids),int);pair_sse=0.
    for start in range(0,len(rows),batch_size):
        batch=rows[start:start+batch_size]
        pred=model(*inputs(data,pack,batch,use_structure)).numpy()+pack['target_mean']
        if not np.isfinite(pred).all():raise ValueError('Nonfinite predictions')
        target_ids=data.target_ids[batch];positions=np.array([lookup[int(i)] for i in target_ids])
        np.add.at(sums,positions,pred);np.add.at(counts,positions,1)
        pair_sse+=float(np.sum((pred.astype(float)-data.target[target_ids])**2))
    return ids,(sums/counts[:,None]).astype(np.float32),float(np.sqrt(pair_sse/(len(rows)*data.target.shape[1])))


def evaluate(data,ids,pred):
    truth=data.target[ids];table=data.vo
    absolute=float(np.sqrt(np.mean((pred.astype(float)-truth)**2)))
    controls=table.loc[table.level.eq('Control')]
    if controls.duplicated(['compound','unit','time_h']).any():raise ValueError('Ambiguous control conditions')
    control_map=controls.set_index(['compound','unit','time_h']).condition_id.to_dict()
    lookup={int(cid):i for i,cid in enumerate(ids)};rows=[]
    for i,cid in enumerate(ids):
        row=table.iloc[cid]
        if row.level=='Control':continue
        control=control_map.get((row.compound,row.unit,row.time_h))
        if control is None or control not in lookup:raise ValueError('Predicted matched control missing')
        observed=data.target[cid]-data.target[control]
        predicted=pred[i]-pred[lookup[control]]
        rows.append(dict(compound=row.compound,condition_id=int(cid),time_h=row.time_h,level=row.level,
            response_mse=float(np.mean((predicted.astype(float)-observed)**2)),zero_response_mse=float(np.mean(observed.astype(float)**2))))
    detail=pd.DataFrame(rows)
    if detail.empty:raise ValueError('No treated target conditions')
    response=float(np.sqrt(detail.response_mse.mean()));zero=float(np.sqrt(detail.zero_response_mse.mean()))
    return dict(absolute_rmse=absolute,response_rmse=response,zero_response_rmse=zero,ratio_to_no_change=response/zero if zero>0 else np.nan,n_target_conditions=len(ids),n_treated_conditions=len(detail)),detail


def run(a,model_factory=None,prepare_fn=None,variants=None):
    if a.out.exists() and any(a.out.iterdir()):raise ValueError('Use a fresh output directory')
    a.out.mkdir(parents=True,exist_ok=True);torch.set_num_threads(a.threads)
    model_factory=model_for if model_factory is None else model_factory
    prepare_fn=prepare if prepare_fn is None else prepare_fn
    data=Data(a);data.structure_audit.to_csv(a.out/'structure_audit.csv',index=False)
    names=np.random.default_rng(a.seed).permutation(data.names)
    n_test=max(1,int(np.ceil(.2*len(names))));test_names=set(names[:n_test])
    remaining=names[n_test:];n_val=max(1,int(np.ceil(.2*len(remaining))))
    val_names=set(remaining[:n_val]);train_names=set(remaining[n_val:])
    if len(train_names)<2:raise ValueError('Too few compounds for 3-way split')
    sets=[train_names,val_names,test_names]
    if any(sets[i]&sets[j] for i in range(3) for j in range(i+1,3)):raise ValueError('Compound split overlap')
    inner_tr=np.flatnonzero(data.pairs.compound.isin(train_names));inner_val=np.flatnonzero(data.pairs.compound.isin(val_names))
    outer_tr=np.flatnonzero(~data.pairs.compound.isin(test_names));outer_te=np.flatnonzero(data.pairs.compound.isin(test_names))
    split=[]
    for stage,values in zip(['inner_train','inner_validation','outer_test'],sets):split.extend(dict(compound=n,stage=stage) for n in sorted(values))
    pd.DataFrame(split).to_csv(a.out/'compound_split.csv',index=False)
    config={k:str(v) if isinstance(v,Path) else v for k,v in vars(a).items()}
    config.update(protocol='same MLP with own Morgan features versus zero-filled structure slots',selection='internal target-averaged absolute MSE',input_policy='each available treated vitro condition predicts each target; average predictions per target for main evaluation',target='condition-mean expression, all probes directly; no target PCA')
    (a.out/'config.json').write_text(json.dumps(config,indent=2))
    print(f'Compounds train/val/test: {list(map(len,sets))}; probes={data.target.shape[1]}',flush=True)
    print('Missing structure compounds:',data.structure_audit.loc[data.structure_audit.structure_present.eq(0),'compound'].tolist(),flush=True)
    inner=prepare_fn(data,inner_tr,inner_val);selected=[];history=[]
    for name,use_structure in (variants if variants is not None else [('expression_only',False),('expression_structure',True)]):
        model=model_factory(data,inner,a.seed)
        opt=torch.optim.AdamW(model.parameters(),lr=.001,weight_decay=.01)
        best=float('inf');best_epoch=None;best_state=None
        for epoch in range(1,a.epochs+1):
            loss=train_epoch(model,opt,data,inner,inner_tr,use_structure,a.batch_size)
            if epoch==1 or epoch%a.check_every==0 or epoch==a.epochs:
                ids,pred,_=predict(model,data,inner,inner_val,use_structure,a.batch_size)
                mse=float(np.mean((pred.astype(float)-data.target[ids])**2))
                if mse<best:best=mse;best_epoch=epoch;best_state=copy.deepcopy(model.state_dict())
                history.append(dict(model=name,epoch=epoch,train_loss=loss,validation_rmse=np.sqrt(mse),best_epoch=best_epoch))
                pd.DataFrame(history).to_csv(a.out/'validation_history.csv',index=False)
                print(f'{name} epoch={epoch}/{a.epochs}, val_RMSE={np.sqrt(mse):.5f}, best_epoch={best_epoch}',flush=True)
        selected.append((name,use_structure,best_epoch,best))
        torch.save(best_state,a.out/f'{name}_inner_best.pt')
    del inner
    outer=prepare_fn(data,outer_tr,outer_te);joblib.dump(outer['preprocess'],a.out/'preprocess.joblib')
    results=[];response_details=[]
    for name,use_structure,epochs,best in selected:
        model=model_factory(data,outer,a.seed);opt=torch.optim.AdamW(model.parameters(),lr=.001,weight_decay=.01)
        print(f'{name}: outer refit {epochs} epochs',flush=True)
        for epoch in range(1,epochs+1):
            loss=train_epoch(model,opt,data,outer,outer_tr,use_structure,a.batch_size)
            if epoch==1 or epoch%10==0 or epoch==epochs:print(f'{name} refit={epoch}/{epochs}',flush=True)
        ids,pred,pair_rmse=predict(model,data,outer,outer_te,use_structure,a.batch_size)
        metrics,detail=evaluate(data,ids,pred)
        metrics.update(model=name,selected_epoch=epochs,validation_rmse=np.sqrt(best),pair_rmse=pair_rmse,parameters=sum(p.numel() for p in model.parameters()))
        detail['model']=name;response_details.append(detail);results.append(metrics)
        np.savez_compressed(a.out/f'{name}_predictions.npz',target_ids=ids,prediction=pred)
        torch.save(model.state_dict(),a.out/f'{name}.pt')
        print(f'{name}: TEST absolute={metrics["absolute_rmse"]:.5f}, response={metrics["response_rmse"]:.5f}',flush=True)
    summary=pd.DataFrame(results);summary.to_csv(a.out/'comparison.csv',index=False)
    pd.concat(response_details,ignore_index=True).to_csv(a.out/'response_condition_metrics.csv',index=False)
    print(summary.to_string(index=False));print('Saved:',a.out,flush=True)
    return summary



from types import SimpleNamespace
first=SimpleNamespace(prepare=prepare,seed_all=seed_all,run=run)
def load_projection(annotation,prepared):
    annotation=Path(annotation)
    provenance=json.loads((annotation/'provenance.json').read_text())
    probes=pd.read_csv(Path(prepared)/'probes.csv')['probe'].astype(str).tolist()
    if provenance['species']!='Rattus norvegicus' or provenance['probe_order']!=probes:raise ValueError('Species/probe order mismatch')
    projection=np.load(annotation/'projection.npy').astype(np.float64)
    pathways=pd.read_csv(annotation/'pathway_audit.csv')
    if projection.shape!=(len(probes),len(pathways)) or len(pathways)==0 or pathways.pathway.duplicated().any():raise ValueError('Pathway layout mismatch')
    if not np.isfinite(projection).all() or (projection<0).any():raise ValueError('Invalid pathway weights')
    np.testing.assert_allclose(projection.sum(0),1.,rtol=0,atol=1e-6)
    return projection,pathways,provenance

def prepare_pathways(data,tr,ev,projection,pathways,provenance):
    pack=first.prepare(data,tr,ev)
    # Gene standardization was fitted to unique TRAINING in vitro source conditions.
    gene_expression=pack['source'].numpy()
    scores=(gene_expression.astype(np.float64)@projection).astype(np.float32)
    unique_sources=np.unique(data.source_ids[tr])
    pathway_scale=StandardScaler().fit(scores[unique_sources])
    normalized=pathway_scale.transform(scores).astype(np.float32)
    pack['source']=torch.cat([pack['source'],torch.tensor(normalized)],dim=1)
    pack['preprocess'].update(pathway_scale=pathway_scale,pathway_projection=projection,pathway_ids=pathways.pathway.tolist(),pathway_provenance=provenance)
    pack['gene_dim']=data.source.shape[1];pack['pathway_dim']=projection.shape[1]
    return pack
pathway=SimpleNamespace(load_projection=load_projection,prepare_pathways=prepare_pathways)
"""Sparse multi-unit pathway representations, contrasted with mean-only and zero inputs."""
import argparse
import json
from pathlib import Path
from types import SimpleNamespace
import uuid
import numpy as np
import pandas as pd
import torch
from torch import nn
from threadpoolctl import threadpool_limits




class PatternEncoder(nn.Module):
    def __init__(self,membership,mode,units=4,output_dim=32):
        super().__init__()
        if mode not in ['none','mean','sparse']:raise ValueError(mode)
        edges=np.argwhere(np.asarray(membership)>0)
        if not len(edges) or (np.bincount(edges[:,1],minlength=membership.shape[1])==0).any():raise ValueError('Empty pathway')
        self.mode=mode;self.n_pathways=membership.shape[1];self.units=units
        self.register_buffer('probe_index',torch.tensor(edges[:,0],dtype=torch.long))
        self.register_buffer('pathway_index',torch.tensor(edges[:,1],dtype=torch.long))
        fan_in=np.bincount(edges[:,1],minlength=self.n_pathways)
        weights=torch.randn(len(edges),units)/torch.tensor(np.sqrt(fan_in[edges[:,1]]),dtype=torch.float32)[:,None]
        self.edge_weights=nn.Parameter(weights)
        self.bias=nn.Parameter(torch.zeros(self.n_pathways,units))
        self.readout=nn.Sequential(nn.Dropout(.1),nn.Linear(self.n_pathways*units,output_dim,bias=False),nn.SiLU())

    def pathway_units(self,genes,mean_scores):
        batch=len(genes)
        if self.mode=='none':return genes.new_zeros(batch,self.n_pathways,self.units)
        # Each edge has its own signed, trainable weight. Only KEGG membership edges exist.
        values=genes[:,self.probe_index] if self.mode=='sparse' else mean_scores[:,self.pathway_index]
        weighted=values[:,:,None]*self.edge_weights[None,:,:]
        aggregated=genes.new_zeros(batch,self.n_pathways,self.units).index_add(1,self.pathway_index,weighted)
        return F.silu(aggregated+self.bias)

    def forward(self,genes,mean_scores):return self.readout(self.pathway_units(genes,mean_scores).flatten(1))


# Explicit functional import keeps the standalone notebook and module equivalent.
import torch.nn.functional as F


class PatternMLP(nn.Module):
    def __init__(self,gene_dim,condition_dim,structure_dim,output_dim,membership,mode):
        super().__init__();self.gene_dim=gene_dim
        self.expression_encoder=nn.Sequential(nn.Linear(gene_dim,128),nn.SiLU(),nn.Dropout(.1),nn.Linear(128,64),nn.SiLU())
        self.structure_encoder=nn.Sequential(nn.Linear(structure_dim,64,bias=False),nn.SiLU(),nn.Dropout(.1),nn.Linear(64,32,bias=False),nn.SiLU())
        self.pathway_encoder=PatternEncoder(membership,mode,units=4,output_dim=32)
        self.decoder=nn.Sequential(nn.Linear(64+32+32+condition_dim,64),nn.SiLU(),nn.Dropout(.1),nn.Linear(64,output_dim))
    def forward(self,source,condition,structure):
        genes=source[:,:self.gene_dim];mean_scores=source[:,self.gene_dim:]
        expression=self.expression_encoder(genes);chemical=self.structure_encoder(structure)
        biological=self.pathway_encoder(genes,mean_scores)
        return self.decoder(torch.cat([expression,chemical,biological,condition],1))


def run_patterns(a):
    projection,pathways,provenance=pathway.load_projection(a.annotation,a.prepared)
    if a.out.exists() and any(a.out.iterdir()):raise ValueError('Use a fresh output directory')
    a.out.mkdir(parents=True,exist_ok=True)
    config={k:str(v) if isinstance(v,Path) else v for k,v in vars(a).items()}
    config.update(structure='enabled for all three models',units_per_pathway=4,pathway_readout=32,
        mean_control='same sparse architecture but every member receives its pathway mean score',
        sparse_input='individual TRAIN-standardized probe expression at membership edges',
        scope='PNET-inspired sparse representation, not a reproduction of P-NET/GEARS')
    (a.out/'config.json').write_text(json.dumps(config,indent=2))
    pathways.to_csv(a.out/'pathway_audit.csv',index=False)
    (a.out/'pathway_provenance.json').write_text(json.dumps(provenance,ensure_ascii=False,indent=2))
    print(f'Pathways={len(pathways)}, units/pathway=4, membership edges={int(np.count_nonzero(projection))}',flush=True)
    tables=[];reference_split=None
    for mode in ['none','mean','sparse']:
        folder=a.out/mode
        values={**vars(a),'out':folder,'pathway_mode':mode,'architecture':'matched_sparse_pathway_units4_readout32_structure_on'}
        group=SimpleNamespace(**values)
        def prep(data,tr,ev):return pathway.prepare_pathways(data,tr,ev,projection,pathways,provenance)
        def factory(data,pack,seed):
            first.seed_all(seed)
            return PatternMLP(data.source.shape[1],pack['conditions'].shape[1],pack['structure'].shape[1],data.target.shape[1],projection,mode)
        print(f'\nPathway mode={mode}, structure fixed ON',flush=True)
        result=first.run(group,model_factory=factory,prepare_fn=prep,variants=[(f'pathway_{mode}',True)])
        result['pathway_mode']=mode;result['structure']=True;tables.append(result)
        split=pd.read_csv(folder/'compound_split.csv').sort_values('compound').reset_index(drop=True)
        if reference_split is None:reference_split=split
        else:pd.testing.assert_frame_equal(reference_split,split)
        pd.concat(tables,ignore_index=True).to_csv(a.out/'comparison.csv',index=False)
    result=pd.concat(tables,ignore_index=True)
    if result.parameters.nunique()!=1:raise ValueError('Parameter counts differ')
    print(result.to_string(index=False));print('Saved:',a.out,flush=True)
    return result




In [ ]:
from datetime import datetime
PREPARED=Path('/home/seowon1211/data1/conditioned_expression/prepared_controls')
STRUCTURE=Path('/home/seowon1211/data1/conditioned_expression/morgan_matched/morgan_binary.csv')
ANNOTATION=Path('/home/seowon1211/data1/conditioned_expression/kegg_rat_20261102_034010_265aed')
OUT=PREPARED.parent / ('pathway_pattern_mlp_'+datetime.now().strftime('%Y%m%d_%H%M%S')+'_'+uuid.uuid4().hex[:6])
args=SimpleNamespace(prepared=PREPARED,structure=STRUCTURE,annotation=ANNOTATION,out=OUT,epochs=30,check_every=5,batch_size=128,threads=4,seed=42)
for path in [PREPARED,STRUCTURE,ANNOTATION]:
    assert path.exists(), f'경로 확인: {path}'
print('출력:',OUT,flush=True)
with threadpool_limits(limits=args.threads):
    comparison=run_patterns(args)
display(comparison)
reference=comparison.set_index('pathway_mode')
for mode in ['mean','sparse']:
    print(mode, 'vs none:', {metric:float(reference.loc[mode,metric]-reference.loc['none',metric]) for metric in ['absolute_rmse','response_rmse']})
print('sparse vs mean:', {metric:float(reference.loc['sparse',metric]-reference.loc['mean',metric]) for metric in ['absolute_rmse','response_rmse']})